# Benchmark analysis, Colab runner

Self-contained. Regenerates every table in the paper and adds the two things
the draft is missing: **bootstrap confidence intervals** and the **figures**.

No model calls, no regeneration. Pure analysis of frozen artifacts.

Runtime about 20 to 30 min on a free CPU runtime. No GPU.

**Run cells top to bottom.** Cell 2 finds your files automatically; it will
tell you if anything is missing before any analysis starts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

# Located by structure, not by folder names, so renaming or removing a level
# in Drive will not break this. A RESULTS folder qualifies only if it actually
# contains EVALUATION-RESULT.
def find_results_root():
    for base in ['/content/drive/Shareddrives', '/content/drive/MyDrive']:
        if not os.path.isdir(base):
            continue
        for hit in glob.glob(f'{base}/**/RESULTS', recursive=True):
            if os.path.isdir(os.path.join(hit, 'EVALUATION-RESULT')):
                return hit
    return None

RESULTS_ROOT = find_results_root()

if RESULTS_ROOT is None:
    print('Could not locate it. Shared drives visible:')
    for d in glob.glob('/content/drive/Shareddrives/*'):
        print('   ', d)
    print('\nSet it by hand, e.g.:')
    print("RESULTS_ROOT = '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS'")
    raise SystemExit('RESULTS folder not found.')

OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
print('RESULTS_ROOT =', RESULTS_ROOT)
print('OUT          =', OUT)
print()
for d in sorted(os.listdir(RESULTS_ROOT)):
    print('  ', d)

## 1. Find and stage the input files

Searches the whole `RESULTS` tree for the files the pipeline needs and copies
them to local disk. Accepts `.xlsx` or exported `.csv` for the rater sheets.

If a required file is missing this cell says so and stops. If several copies
exist it takes the largest, which avoids picking up a truncated or placeholder
version.

In [ ]:
import os, shutil, glob

RAW = '/content/raw'
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)
os.environ['HALLUBENCH_RAW'] = RAW
os.environ['HALLUBENCH_OUT'] = OUT

E = f'{RESULTS_ROOT}/EVALUATION-RESULT'
HA = f'{RESULTS_ROOT}/HALLUCINATION-RESULT'
AN = f'{RESULTS_ROOT}/ANNOTATION-RESULT'
RA = f'{RESULTS_ROOT}/RATERS-RESULT/Raters-scores'   # NOT .../old/

# Exact locations, read off the Drive folders.
EXPECTED = {
    'panel_raw_judge_labels_full.csv': (f'{E}/panel_raw_judge_labels_full.csv', True),
    'full_labeled_dataset_full.csv':   (f'{E}/full_labeled_dataset_full.csv',   True),
    'Rater-A-scores-807.xlsx':         (f'{RA}/Rater-A-scores-807.xlsx',        True),
    'Rater-B-scores-807.xlsx':         (f'{RA}/Rater-B-scores-807.xlsx',        True),
    'Panel-A-scores-807.xlsx':         (f'{RA}/Panel-A-scores-807.xlsx',        True),
    'embeddings_openai.npy':           (f'{HA}/embeddings_openai.npy',          False),
    'embedding_index.csv':             (f'{HA}/embedding_index.csv',            False),
    'UCFCrime_Train.json':             (f'{AN}/UCFCrime_Train.json',            False),
    'UCFCrime_Val.json':               (f'{AN}/UCFCrime_Val.json',              False),
    'UCFCrime_Test.json':              (f'{AN}/UCFCrime_Test.json',             False),
}

def fallback(name):
    """Search the tree if the expected path moved. Never looks inside old/."""
    hits = [p for p in glob.glob(f'{RESULTS_ROOT}/**/{name}', recursive=True)
            if os.path.isfile(p) and '/old/' not in p.replace(os.sep, '/')]
    return max(hits, key=os.path.getsize) if hits else None

missing = []
for name, (path, required) in EXPECTED.items():
    src = path if os.path.isfile(path) else fallback(name)
    if src is None:
        missing.append((name, required))
        print(f'  MISSING {"(required)" if required else "(optional)"}  {name}')
        continue
    shutil.copy(src, os.path.join(RAW, name))
    size = os.path.getsize(os.path.join(RAW, name))
    note = ''
    if size < 1024:
        note = '   <-- EMPTY OR PLACEHOLDER, do not proceed'
    elif src != path:
        note = f'   (found at {src})'
    print(f'  ok  {name:34s} {size/1e6:9.2f} MB{note}')

req = [n for n, r in missing if r]
if req:
    raise SystemExit(f'\nSTOP: required files not found: {req}')
print('\nstaged ->', RAW)

# Sanity: these are the sizes to expect.
print('\nExpected roughly: panel_raw ~15 MB, full_labeled ~200 MB, '
      'embeddings ~119 MB, rater sheets ~20-60 KB each.')

### Rater sheets exported as CSV?

If the cell above staged `Rater-A-scores-807.csv` rather than `.xlsx`, run this
to convert them back so the scripts find what they expect. Harmless if not
needed.

In [ ]:
import pandas as pd, os
for n in ['Rater-A-scores-807', 'Rater-B-scores-807', 'Panel-A-scores-807']:
    csv, xls = f'{RAW}/{n}.csv', f'{RAW}/{n}.xlsx'
    if os.path.exists(csv) and not os.path.exists(xls):
        pd.read_csv(csv).to_excel(xls, index=False)
        print('converted', n)
print('done')

In [ ]:
!pip -q install openpyxl tabulate

## 2. Write the pipeline scripts

Embedded here so nothing needs uploading.

In [ ]:
!mkdir -p /content/scripts
%cd /content/scripts

In [ ]:
%%writefile config.py
"""
Shared configuration, paths and helpers.

Set the input directory once, either by editing RAW_DIR below or by exporting
an environment variable before running anything:

    export HALLUBENCH_RAW=/path/to/your/files
    export HALLUBENCH_OUT=/path/to/write/outputs

Expected files in RAW_DIR (names as released):

    panel_raw_judge_labels_full.csv      per-judge H1-H6, 2 judges x 19,361
    full_labeled_dataset_full.csv        report text + ground truth (TEXT ONLY)
    embeddings_openai.npy                19,361 x 1536, optional
    embedding_index.csv                  row order for the .npy, optional
    Rater-A-scores-807.xlsx              human rater A, 130 reports
    Rater-B-scores-807.xlsx              human rater B, 130 reports
    Panel-A-scores-807.xlsx              panel labels for the same 130
    UCFCrime_Train.json / _Val / _Test   UCA annotations, optional (audit only)

IMPORTANT: full_labeled_dataset_full.csv aggregates the two judges with OR and
does not reproduce the published statistics. It is used here only as a source
of report text. All labels are rebuilt from panel_raw_judge_labels_full.csv
under the paper's rule: two-judge majority, ties broken toward no
hallucination (with two judges, AND).
"""
import os
import re
from pathlib import Path

RAW_DIR = Path(os.environ.get("HALLUBENCH_RAW", "./raw"))
OUT_DIR = Path(os.environ.get("HALLUBENCH_OUT", "./out"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

H = ["H1", "H2", "H3", "H4", "H5", "H6"]
TARGETS = H + ["any_hallucination"]
KEY = ["model", "technique", "video", "crime_type"]
JOIN = ["model", "technique", "video"]

HNAME = {"H1": "Scene Fabrication", "H2": "Crime Misclassification",
         "H3": "Crime Missed", "H4": "Severity Minimization",
         "H5": "Entity Fabrication", "H6": "Phantom Actors"}
AXIS_OF = {"H1": "fabrication", "H5": "fabrication", "H6": "fabrication",
           "H3": "omission", "H2": "distortion", "H4": "distortion"}
AXES = {"fabrication": ["H1", "H5", "H6"], "omission": ["H3"],
        "distortion": ["H2", "H4"]}

MULTI_TURN = {"Sequential", "ReAct", "Least-to-Most", "True-Iterative"}
SINGLE_TURN = {"Zero-Shot", "Chain-of-Thought", "Meta-Prompting",
               "Self-Consistency"}

JUDGE_FAIL = -1          # sentinel in the raw judge file; not a verdict
TRUNCATION_CAP = 3000    # judge input cap, in characters

BENCH_FILE = OUT_DIR / "benchmark_labels.csv.gz"

STOPWORDS = set("""a an the and or but if while of to in on at by for with from as is are
was were be been being it its this that these those there here he she they them his her
their you we i not no than then so such which who whom what when where how also into over
under after before up down out off again more most very can will just do does did done
have has had having""".split())

HEDGE_RE = re.compile(
    r"\b(may|might|could|possibly|perhaps|appears?|seems?|likely|unclear|uncertain|"
    r"presumably|apparently|suggests?|potentially|probably|cannot determine|"
    r"difficult to)\b", re.I)


def require(*names):
    """Fail early and clearly if an expected input file is absent."""
    missing = [n for n in names if not (RAW_DIR / n).exists()]
    if missing:
        raise SystemExit(
            f"Missing input file(s) in {RAW_DIR.resolve()}:\n  "
            + "\n  ".join(missing)
            + "\n\nSet HALLUBENCH_RAW to the directory holding them."
        )
    return [RAW_DIR / n for n in names]


def content_tokens(s):
    """Lowercase content words, stopwords and short tokens removed."""
    return [w for w in re.findall(r"[a-z]+", str(s).lower())
            if w not in STOPWORDS and len(w) > 2]


def load_benchmark():
    """Load the rebuilt benchmark table, with a clear error if not built yet."""
    import pandas as pd
    if not BENCH_FILE.exists():
        raise SystemExit(f"{BENCH_FILE} not found. Run 00_build_benchmark.py first.")
    df = pd.read_csv(BENCH_FILE)
    df["model_output"] = df.model_output.fillna("")
    df["ground_truth"] = df.ground_truth.fillna("")
    return df


def banner(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


In [ ]:
%%writefile 00_build_benchmark.py
#!/usr/bin/env python3
"""
Step 0 - rebuild canonical labels and construct video-disjoint splits.

Labels come from panel_raw_judge_labels_full.csv under the paper's rule:
two-judge majority, ties broken toward no hallucination. Reproduces the
published corpus statistics (ANY 91.1%, 2.58 hallucinations per report).

Writes: benchmark_labels.csv.gz, label_reliability_tiers.csv,
        benchmark_manifest.json, INTEGRITY_REPORT.md
"""
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

from config import (H, KEY, JOIN, AXIS_OF, AXES, SEED, RAW_DIR, OUT_DIR,
                    JUDGE_FAIL, MULTI_TURN, require, banner)

PAPER = {"any": 91.1, "mean": 2.58}
log_lines = []


def log(m=""):
    print(m)
    log_lines.append(m)


require("panel_raw_judge_labels_full.csv", "full_labeled_dataset_full.csv",
        "Rater-A-scores-807.xlsx", "Rater-B-scores-807.xlsx",
        "Panel-A-scores-807.xlsx")

banner("STEP 0  build benchmark")

# ------------------------------------------------------------ integrity
log("## 1. Source integrity\n")
raw = pd.read_csv(RAW_DIR / "panel_raw_judge_labels_full.csv")
log(f"- raw judge rows: {len(raw):,}")
log(f"- reports: {raw.groupby(KEY).ngroups:,}")
log(f"- judges per report: {raw.groupby(KEY).size().value_counts().to_dict()}")
log(f"- self-judging rows (judge == generator): {(raw.judge == raw.model).sum()}")

sent = (raw[H] == JUDGE_FAIL)
n_cells, n_rows = int(sent.sum().sum()), int(sent.any(axis=1).sum())
log(f"- judge-failure sentinels ({JUDGE_FAIL}): {n_cells} cells / {n_rows} rows "
    f"-> treated as missing, excluded from the vote")
raw[H] = raw[H].mask(sent)

# ---------------------------------------------------------- aggregation
log("\n## 2. Label aggregation (two-judge majority, ties -> negative)\n")
grp = raw.groupby(KEY)[H]
pos, n = grp.sum(min_count=1), grp.count()
lab = (pos >= 2).astype(int)
short = n < 2                                   # a judge failed on this type
lab[short] = (pos[short] >= n[short]).astype(int)
lab = lab.reset_index()

lab["hallucination_count"] = lab[H].sum(axis=1)
lab["any_hallucination"] = (lab.hallucination_count > 0).astype(int)
for ax, cols in AXES.items():
    lab[f"axis_{ax}"] = lab[cols].max(axis=1)

got = (100 * lab.any_hallucination.mean(), lab.hallucination_count.mean())
log(f"- ANY hallucination: {got[0]:.2f}%   (paper {PAPER['any']}%)")
log(f"- mean per report:   {got[1]:.3f}   (paper {PAPER['mean']})")
if abs(got[0] - PAPER["any"]) > 0.5 or abs(got[1] - PAPER["mean"]) > 0.05:
    log("  !! WARNING: does not match the published statistics. Check the "
        "aggregation rule and the input file.")
log("\nPer-model per-type rate (%):\n")
log((100 * lab.groupby("model")[H].mean()).round(1).to_markdown())

# ----------------------------------------------------------- join text
txt = pd.read_csv(RAW_DIR / "full_labeled_dataset_full.csv",
                  usecols=KEY + ["ground_truth", "model_output",
                                 "model_output_full_len"])
bench = lab.merge(txt, on=KEY, how="left", validate="one_to_one")
assert bench.model_output.notna().all(), "text join incomplete"

# --------------------------------------------------- reliability tiers
log("\n## 3. Label reliability against human raters (n=130)\n")
ra = pd.read_excel(RAW_DIR / "Rater-A-scores-807.xlsx")
rb = pd.read_excel(RAW_DIR / "Rater-B-scores-807.xlsx")
pa = pd.read_excel(RAW_DIR / "Panel-A-scores-807.xlsx")

rows = []
tiers = {}
for h in H:
    a, b = ra[f"human_{h}"].to_numpy(), rb[f"human_{h}"].to_numpy()
    pan = pa[f"panel_{h}"].to_numpy()
    agreed = a == b                     # paper's consensus definition
    k_hh = cohen_kappa_score(a, b)
    k_pc = cohen_kappa_score(a[agreed], pan[agreed])
    tier = "high" if k_pc >= 0.65 else ("medium" if k_pc >= 0.40 else "low")
    tiers[h] = tier
    rows.append({"type": h, "axis": AXIS_OF[h], "n_consensus": int(agreed.sum()),
                 "kappa_human_human": round(k_hh, 3),
                 "kappa_panel_consensus": round(k_pc, 3), "tier": tier})
tier_df = pd.DataFrame(rows)
log(tier_df.to_markdown(index=False))
log(f"\n- macro kappa: {tier_df.kappa_panel_consensus.mean():.3f}  (paper 0.556)")
log("- NOTE: these kappas are computed only on rows where both raters agree; "
    "n varies by type and is reported above.")
log(f"- low-reliability types (exclude from headline macro): "
    f"{[h for h in H if tiers[h] == 'low']}")

gold = pa[JOIN].copy()
gold["gold"] = 1
bench = bench.merge(gold, on=JOIN, how="left")
bench["gold"] = bench.gold.fillna(0).astype(int)

# --------------------------------------------------------------- splits
log("\n## 4. Splits (video-disjoint)\n")
rng = np.random.default_rng(SEED)
vmeta = bench[["video", "crime_type"]].drop_duplicates().sort_values("video")
assign = {}
for ct, g in vmeta.groupby("crime_type"):
    v = g.video.to_numpy().copy()
    rng.shuffle(v)
    n_tr, n_va = int(0.70 * len(v)), int(0.15 * len(v))
    for i, name in ((slice(0, n_tr), "train"),
                    (slice(n_tr, n_tr + n_va), "val"),
                    (slice(n_tr + n_va, None), "test")):
        assign.update({x: name for x in v[i]})
bench["split_random"] = bench.video.map(assign)
bench["split_heldout_model"] = np.where(bench.model == "Gemini", "test", "train")
bench["split_heldout_technique"] = np.where(
    bench.technique.isin(MULTI_TURN), "test", "train")

for c in ["split_random", "split_heldout_model", "split_heldout_technique"]:
    leak = int(bench.groupby("video")[c].nunique().gt(1).sum())
    log(f"- {c}: {bench[c].value_counts().to_dict()}   "
        f"videos spanning >1 fold: {leak}")
log(f"\n- gold (human-validated) reports: {int(bench.gold.sum())}")

# ---------------------------------------------------------------- write
cols = (KEY + H + ["hallucination_count", "any_hallucination"]
        + [f"axis_{a}" for a in AXES]
        + ["gold", "split_random", "split_heldout_model",
           "split_heldout_technique", "model_output_full_len",
           "ground_truth", "model_output"])
bench[cols].to_csv(OUT_DIR / "benchmark_labels.csv.gz", index=False,
                   compression="gzip")
tier_df.to_csv(OUT_DIR / "label_reliability_tiers.csv", index=False)
json.dump({"seed": SEED,
           "aggregation": "two-judge majority, ties -> negative",
           "n_reports": int(len(bench)), "n_videos": int(bench.video.nunique()),
           "any_rate": round(float(bench.any_hallucination.mean()), 4),
           "mean_per_report": round(float(bench.hallucination_count.mean()), 4),
           "judge_failure_cells": n_cells, "reliability_tiers": tiers},
          open(OUT_DIR / "benchmark_manifest.json", "w"), indent=2)
(OUT_DIR / "INTEGRITY_REPORT.md").write_text(
    "# Benchmark rebuild integrity report\n\n" + "\n".join(log_lines) + "\n")
print(f"\nwrote -> {OUT_DIR}")


In [ ]:
%%writefile 01_baselines.py
#!/usr/bin/env python3
"""
Step 1 - baseline detectors.

Feature sets: surface, surface_noid (identity removed), tfidf.
Splits: random, heldout_model, heldout_technique.

Generator identity is dropped automatically on heldout_model and technique
identity on heldout_technique, since those features are constant in training
and unseen at test.

Writes: baseline_results.csv, BASELINE_REPORT.md
"""
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from config import (H, TARGETS, SEED, OUT_DIR, MULTI_TURN, HEDGE_RE,
                    TRUNCATION_CAP, load_benchmark, banner)

warnings.filterwarnings("ignore")
banner("STEP 1  baselines")
df = load_benchmark()

txt = df.model_output
w = txt.str.split().str.len().clip(lower=1)
s = txt.str.count(r"[.!?]").clip(lower=1)
surf = pd.DataFrame({
    "n_words": w, "log_words": np.log1p(w), "n_sents": s,
    "mean_sent_len": w / s,
    "hedge_rate": txt.str.count(HEDGE_RE) / w,
    "detail_rate": txt.str.count(r"\b(\d+|\d{1,2}:\d{2})\b") / w,
    "type_token": txt.str.lower().apply(
        lambda x: len(set(x.split())) / max(len(x.split()), 1)),
    "truncated": (df.model_output_full_len > TRUNCATION_CAP).astype(int),
    "gt_words": df.ground_truth.str.split().str.len(),
})
id_gen = pd.get_dummies(df.model, prefix="gen").astype(float)
id_tec = pd.get_dummies(df.technique, prefix="tec").astype(float)
id_tec["multi_turn"] = df.technique.isin(MULTI_TURN).astype(float)

print("fitting tf-idf ...")
X_tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                          sublinear_tf=True,
                          strip_accents="unicode").fit_transform(txt)

SPLITS = {"random": ("split_random", True, True),
          "heldout_model": ("split_heldout_model", False, True),
          "heldout_technique": ("split_heldout_technique", True, False)}


def build(fs, use_gen, use_tec):
    if fs == "tfidf":
        return X_tfidf
    parts = [surf.to_numpy(float)]
    if fs == "surface":
        if use_gen:
            parts.append(id_gen.to_numpy())
        if use_tec:
            parts.append(id_tec.to_numpy())
    return np.hstack(parts)


rows = []
for sname, (scol, ug, ut) in SPLITS.items():
    tr, te = (df[scol] == "train").to_numpy(), (df[scol] == "test").to_numpy()
    for fs in ["surface", "surface_noid", "tfidf"]:
        X = build(fs, ug, ut)
        if sparse.issparse(X):
            Xtr, Xte = X[tr], X[te]
        else:
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        for t in TARGETS:
            y = df[t].to_numpy()
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                rows.append({"split": sname, "features": fs, "target": t,
                             "auc": np.nan, "f1": np.nan}); continue
            clf = LogisticRegression(max_iter=2000, class_weight="balanced",
                                     random_state=SEED).fit(Xtr, y[tr])
            p = clf.predict_proba(Xte)[:, 1]
            rows.append({"split": sname, "features": fs, "target": t,
                         "auc": roc_auc_score(y[te], p),
                         "f1": f1_score(y[te], (p >= .5).astype(int)),
                         "pos_rate_test": float(y[te].mean())})
        print(f"  {sname} / {fs}")

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "baseline_results.csv", index=False)

out = ["# Baseline sweep\n",
       "Logistic regression, balanced class weights, seed 42. "
       "All splits video-disjoint. `macro_hq` excludes H1 (low tier).\n"]
for fs in ["surface", "surface_noid", "tfidf"]:
    out.append(f"\n## {fs}\n")
    p = res[res.features == fs].pivot(index="split", columns="target",
                                      values="auc")[TARGETS].round(3)
    p["macro"] = p[H].mean(axis=1).round(3)
    p["macro_hq"] = p[[h for h in H if h != "H1"]].mean(axis=1).round(3)
    out.append(p.to_markdown())
(OUT_DIR / "BASELINE_REPORT.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
%%writefile 02_logo_grounded.py
#!/usr/bin/env python3
"""
Step 2 - leave-one-generator-out, with grounded and embedding baselines.

Three folds (hold out Claude, GPT, Gemini in turn). Feature sets:
  tfidf              lexical content, ungrounded
  embed              OpenAI embeddings, ungrounded  (skipped if .npy absent)
  grounded           report-vs-reference features only
  grounded+surface   grounded plus the surface block

Writes: logo_results.csv, LOGO_REPORT.md
"""
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from config import (H, TARGETS, AXES, SEED, RAW_DIR, OUT_DIR, JOIN, HEDGE_RE,
                    TRUNCATION_CAP, content_tokens, load_benchmark, banner)

warnings.filterwarnings("ignore")
banner("STEP 2  leave-one-generator-out")
df = load_benchmark()

print("grounded features ...")
rt = [set(content_tokens(x)) for x in df.model_output]
gt = [set(content_tokens(x)) for x in df.ground_truth]
G = pd.DataFrame([{
    "novel_rate": len(r - q) / max(len(r), 1),
    "missing_rate": len(q - r) / max(len(q), 1),
    "jaccard": len(r & q) / max(len(r | q), 1),
    "ref_coverage": len(r & q) / max(len(q), 1),
    "len_ratio": len(r) / max(len(q), 1),
    "log_novel": np.log1p(len(r - q)),
    "log_missing": np.log1p(len(q - r)),
} for r, q in zip(rt, gt)])

tf = TfidfVectorizer(max_features=30000, min_df=3, sublinear_tf=True,
                     strip_accents="unicode", stop_words="english")
tf.fit(pd.concat([df.model_output, df.ground_truth.drop_duplicates()]))
A, B = tf.transform(df.model_output), tf.transform(df.ground_truth)
num = np.asarray(A.multiply(B).sum(axis=1)).ravel()
den = np.sqrt(np.asarray(A.multiply(A).sum(axis=1)).ravel()
              * np.asarray(B.multiply(B).sum(axis=1)).ravel()) + 1e-9
G["tfidf_cos"] = num / den

w = df.model_output.str.split().str.len().clip(lower=1)
S = pd.DataFrame({"n_words": w, "log_words": np.log1p(w),
                  "hedge_rate": df.model_output.str.count(HEDGE_RE) / w,
                  "digit_rate": df.model_output.str.count(r"\d") / w,
                  "truncated": (df.model_output_full_len > TRUNCATION_CAP).astype(int)})

X_tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                          sublinear_tf=True, strip_accents="unicode"
                          ).fit_transform(df.model_output)

FEATS = {"tfidf": lambda: X_tfidf,
         "grounded": lambda: G.to_numpy(float),
         "grounded+surface": lambda: np.hstack([G.to_numpy(float),
                                                S.to_numpy(float)])}

emb_ok = (RAW_DIR / "embeddings_openai.npy").exists() and \
         (RAW_DIR / "embedding_index.csv").exists()
if emb_ok:
    E = np.load(RAW_DIR / "embeddings_openai.npy")
    ix = pd.read_csv(RAW_DIR / "embedding_index.csv")
    ix["emb_row"] = np.arange(len(ix))
    d2 = df.merge(ix[JOIN + ["emb_row"]], on=JOIN, how="left",
                  validate="one_to_one")
    assert d2.emb_row.notna().all(), "embedding index misses reports"
    E = E[d2.emb_row.to_numpy().astype(int)]
    FEATS["embed"] = lambda: E
    print(f"embeddings aligned: {E.shape}")
else:
    print("embeddings not found -> skipping the `embed` baseline")

rows = []
from scipy import sparse
for fname, build in FEATS.items():
    X = build()
    for held in ["Claude", "GPT", "Gemini"]:
        tr, te = (df.model != held).to_numpy(), (df.model == held).to_numpy()
        if sparse.issparse(X):
            Xtr, Xte = X[tr], X[te]
        else:
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        for t in TARGETS:
            y = df[t].to_numpy()
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                rows.append({"features": fname, "held_out": held,
                             "target": t, "auc": np.nan}); continue
            clf = LogisticRegression(max_iter=3000, class_weight="balanced",
                                     random_state=SEED).fit(Xtr, y[tr])
            rows.append({"features": fname, "held_out": held, "target": t,
                         "auc": roc_auc_score(y[te],
                                              clf.predict_proba(Xte)[:, 1])})
        print(f"  {fname} / held-out {held}")

r = pd.DataFrame(rows)
r.to_csv(OUT_DIR / "logo_results.csv", index=False)

piv = r.pivot_table(index="features", columns="target",
                    values="auc")[TARGETS].round(3)
for ax, cs in AXES.items():
    piv[ax] = piv[cs].mean(axis=1).round(3)
piv["macro"] = piv[H].mean(axis=1).round(3)
out = ["# Leave-one-generator-out\n",
       "AUC on the held-out generator, averaged over three folds.\n",
       piv.to_markdown(), "\n\n## Per-fold detail\n"]
for f in FEATS:
    out.append(f"\n### {f}\n")
    out.append(r[r.features == f].pivot(index="held_out", columns="target",
                                        values="auc")[TARGETS].round(3).to_markdown())
(OUT_DIR / "LOGO_REPORT.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
%%writefile 03_artifact_test.py
#!/usr/bin/env python3
"""
Step 3 - label-source divergence test.

For each candidate surface feature, compare how well it predicts the PANEL
label against how well it predicts the HUMAN label, on the same 130 reports.
A large positive gap means the feature tracks the judge's decision rule rather
than the phenomenon the humans are scoring.

Also reports panel over-flagging and misses against agreed human labels.

Writes: artifact_test.csv, ARTIFACT_TEST.md
"""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from config import (H, HNAME, RAW_DIR, OUT_DIR, JOIN, HEDGE_RE,
                    content_tokens, load_benchmark, require, banner)

require("Rater-A-scores-807.xlsx", "Rater-B-scores-807.xlsx",
        "Panel-A-scores-807.xlsx")
banner("STEP 3  artifact test")

df = load_benchmark()
ra = pd.read_excel(RAW_DIR / "Rater-A-scores-807.xlsx")
rb = pd.read_excel(RAW_DIR / "Rater-B-scores-807.xlsx")
pa = pd.read_excel(RAW_DIR / "Panel-A-scores-807.xlsx")

g = pa[JOIN].copy()
for h in H:
    a, b = ra[f"human_{h}"].to_numpy(), rb[f"human_{h}"].to_numpy()
    g[f"panel_{h}"] = pa[f"panel_{h}"].to_numpy()
    g[f"human_{h}"] = a
    g[f"agree_{h}"] = (a == b).astype(int)

m = g.merge(df[JOIN + ["model_output", "ground_truth", "model_output_full_len"]],
            on=JOIN, how="left", validate="one_to_one")
assert m.model_output.notna().all(), "gold reports not found in benchmark table"

rt = [set(content_tokens(x)) for x in m.model_output]
gt = [set(content_tokens(x)) for x in m.ground_truth]
FEATURES = {
    "n_words": m.model_output.str.split().str.len().to_numpy(float),
    "novel_rate": np.array([len(r - q) / max(len(r), 1) for r, q in zip(rt, gt)]),
    "missing_rate": np.array([len(q - r) / max(len(q), 1) for r, q in zip(rt, gt)]),
    "hedge_rate": (m.model_output.str.count(HEDGE_RE)
                   / m.model_output.str.split().str.len().clip(lower=1)).to_numpy(),
}

rows = []
for fname, x in FEATURES.items():
    for h in H:
        yp = m[f"panel_{h}"].to_numpy()
        ok = m[f"agree_{h}"].to_numpy() == 1
        yh = m[f"human_{h}"].to_numpy()[ok]
        ap = roc_auc_score(yp, x) if len(np.unique(yp)) > 1 else np.nan
        ah = roc_auc_score(yh, x[ok]) if len(np.unique(yh)) > 1 else np.nan
        rows.append({"feature": fname, "type": h, "name": HNAME[h],
                     "n_human_agreed": int(ok.sum()),
                     "panel_pos_rate": round(float(yp.mean()), 3),
                     "human_pos_rate": round(float(yh.mean()), 3),
                     "AUC_vs_panel": round(ap, 3), "AUC_vs_human": round(ah, 3),
                     "gap": round(ap - ah, 3)})
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "artifact_test.csv", index=False)

out = ["# Label-source divergence\n",
       "AUC of each single feature against the panel label and against the "
       "human label, on the 130 human-validated reports. The human column is "
       "restricted to rows where both raters agree; n is reported per type.\n",
       "A large positive gap means the feature predicts the judge better than "
       "it predicts the humans.\n"]
for fname in FEATURES:
    out.append(f"\n## {fname}\n")
    out.append(res[res.feature == fname][
        ["type", "name", "n_human_agreed", "panel_pos_rate", "human_pos_rate",
         "AUC_vs_panel", "AUC_vs_human", "gap"]].to_markdown(index=False))

out.append("\n\n## Panel behaviour against agreed human labels\n")
rows3 = []
for h in H:
    neg = (m[f"agree_{h}"] == 1) & (m[f"human_{h}"] == 0)
    pos = (m[f"agree_{h}"] == 1) & (m[f"human_{h}"] == 1)
    rows3.append({
        "type": h, "name": HNAME[h],
        "n_human_neg": int(neg.sum()),
        "over_flag_rate": round(float(m.loc[neg, f"panel_{h}"].mean()), 3) if neg.sum() else None,
        "n_human_pos": int(pos.sum()),
        "miss_rate": round(1 - float(m.loc[pos, f"panel_{h}"].mean()), 3) if pos.sum() else None})
out.append(pd.DataFrame(rows3).to_markdown(index=False))
out.append("\n\nCAVEAT: n per cell is small (9 to 110). Treat any single gap "
           "below ~0.10 as noise.")
(OUT_DIR / "ARTIFACT_TEST.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
%%writefile 04_gt_audit.py
#!/usr/bin/env python3
"""
Step 4 - ground-truth audit sample.

UCA annotations are inherited from UCF-Crime's category labels, and at least
one (Abuse002_x264) carries an annotation describing a road-traffic incident.
This script does not decide correctness; it draws a stratified sample for a
human to check, and flags candidates using a crude keyword heuristic whose
false-positive rate is high by design (it is a triage aid, not a measurement).

Writes: gt_audit_sample.csv  (fill in the `verdict` column by hand)
"""
import ast
import json
import numpy as np
import pandas as pd

from config import RAW_DIR, OUT_DIR, SEED, require, banner

banner("STEP 4  ground-truth audit sample")
files = [f for f in ["UCFCrime_Train.json", "UCFCrime_Val.json",
                     "UCFCrime_Test.json"] if (RAW_DIR / f).exists()]
if not files:
    raise SystemExit("No UCFCrime_*.json found in RAW_DIR; skipping audit.")

recs = {}
for f in files:
    d = json.load(open(RAW_DIR / f))
    for vid, v in d.items():
        if isinstance(v, str):
            v = ast.literal_eval(v)
        recs[vid] = {"video": vid,
                     "crime_type": v.get("crime_type", vid.rstrip("0123456789_x264")),
                     "text": " ".join(v.get("sentences", []))}
gtdf = pd.DataFrame(recs.values())
print(f"loaded {len(gtdf)} annotations from {files}")

KW = {
 "Abuse": ["abuse","beat","hit","slap","kick","punch","push","shov","strangl","drag","throw"],
 "Assault": ["assault","attack","fight","punch","beat","hit","kick","knock","shov"],
 "Burglary": ["burglar","break","broke","pry","climb","window","intrud","steal","stole","door"],
 "Explosion": ["explo","blast","fire","smoke","flame","burst","bomb"],
 "Fighting": ["fight","punch","kick","brawl","beat","hit","wrestl","knock"],
 "RoadAccidents": ["car","vehicle","road","crash","collid","collision","motorcycle","truck","traffic","intersection"],
 "Robbery": ["rob","gun","knife","threat","snatch","demand","cash","money","register"],
 "Shooting": ["shoot","shot","gun","pistol","fire","weapon"],
 "Shoplifting": ["shop","store","shelf","pocket","conceal","steal","stole","merchandise","cashier"],
 "Stealing": ["steal","stole","theft","took","grab","snatch","bag","wallet","pick"],
 "Vandalism": ["vandal","smash","break","broke","destroy","damage","graffiti","kick","shatter"],
}
gtdf["flagged"] = [
    0 if any(k in t.lower() for k in KW.get(c, [])) else 1
    for c, t in zip(gtdf.crime_type, gtdf.text)]
print("\nheuristic flag rate by crime type (HIGH false-positive rate, triage only):")
print((gtdf.groupby("crime_type").flagged.agg(["sum", "count"])
       .assign(pct=lambda d: (100 * d["sum"] / d["count"]).round(0))).to_markdown())

rng = np.random.default_rng(SEED)
flagged = gtdf[gtdf.flagged == 1]
clean = gtdf[gtdf.flagged == 0]
samp = pd.concat([
    flagged.sample(min(25, len(flagged)), random_state=SEED),
    clean.sample(min(25, len(clean)), random_state=SEED)]).sample(frac=1, random_state=SEED)
samp = samp[["video", "crime_type", "flagged", "text"]].copy()
samp["text"] = samp.text.str.slice(0, 600)
samp["verdict"] = ""          # annotator fills: match / mismatch / unclear
samp["notes"] = ""
samp.to_csv(OUT_DIR / "gt_audit_sample.csv", index=False)
print(f"\nwrote {len(samp)} rows -> {OUT_DIR / 'gt_audit_sample.csv'}")
print("Fill the `verdict` column (match / mismatch / unclear). The sample is "
      "balanced 25 flagged / 25 unflagged so you can estimate both error "
      "directions, not just the flagged ones.")


In [ ]:
%%writefile 05_judge_confound.py
#!/usr/bin/env python3
"""
Step 5 - judge-pair confound test.

Because no model judges its own output, each generator is scored by a fixed
pair of judges, and those pairs differ sharply in mutual agreement. Under a
conjunctive aggregation rule a disagreeing pair yields fewer positives for
mechanical reasons, so aggregated per-generator rates confound the generator
with its assigned judge pair.

Each judge scores exactly two generators, which permits a within-judge
comparison that holds the judge fixed:

    judge Claude  sees  GPT, Gemini
    judge GPT     sees  Claude, Gemini
    judge Gemini  sees  Claude, GPT

If the generator ordering reported in the aggregated labels is a property of
the generators, every judge should reproduce it on the pair it sees. If it is
a property of the judge pairs, the within-judge orderings will disagree.

Pure analysis of the released per-judge verdicts. No generation, no
re-judging.

Writes: judge_confound.csv, JUDGE_CONFOUND.md
"""
import itertools

import numpy as np
import pandas as pd

from config import H, HNAME, RAW_DIR, OUT_DIR, KEY, JUDGE_FAIL, require, banner

require("panel_raw_judge_labels_full.csv")
banner("STEP 5  judge-pair confound test")

raw = pd.read_csv(RAW_DIR / "panel_raw_judge_labels_full.csv")
raw[H] = raw[H].mask(raw[H] == JUDGE_FAIL)

out = ["# Judge-pair confound test\n",
       "Each generator is scored by a fixed pair of judges. This tests "
       "whether the generator ordering survives when the judge is held "
       "fixed.\n"]

# ---------------------------------------------------- 1. the confound
out.append("\n## 1. Which pair scores which generator\n")
pairs = (raw.groupby("model").judge.unique()
         .apply(lambda a: " + ".join(sorted(a))).rename("judged_by"))
n_rep = raw.groupby("model").size().div(2).astype(int).rename("n_reports")
out.append(pd.concat([pairs, n_rep], axis=1).to_markdown())

# ------------------------------------- 2. aggregated (confounded) view
agg = raw.groupby(KEY)[H]
pos, n = agg.sum(min_count=1), agg.count()
lab = (pos >= 2).astype(int)
short = n < 2
lab[short] = (pos[short] >= n[short]).astype(int)
lab = lab.reset_index()
conf = (100 * lab.groupby("model")[H].mean()).round(1)
out.append("\n\n## 2. Aggregated per-generator rates (confounded)\n")
out.append(conf.to_markdown())

# ------------------------------------------ 3. within-judge comparison
out.append("\n\n## 3. Within-judge rates (%): each judge's own verdicts\n")
wj = (100 * raw.groupby(["judge", "model"])[H].mean()).round(1)
out.append(wj.to_markdown())

# ------------------------------------------------ 4. ordering agreement
out.append("\n\n## 4. Does each judge reproduce the aggregated ordering?\n")
out.append("For every judge and every type, the ordering of the two "
           "generators that judge sees, compared against the ordering the "
           "aggregated labels give for the same two generators.\n")
rows = []
for judge in sorted(raw.judge.unique()):
    seen = sorted(raw.loc[raw.judge == judge, "model"].unique())
    for a, b in itertools.combinations(seen, 2):
        for h in H:
            wa = raw.loc[(raw.judge == judge) & (raw.model == a), h].mean()
            wb = raw.loc[(raw.judge == judge) & (raw.model == b), h].mean()
            ca = lab.loc[lab.model == a, h].mean()
            cb = lab.loc[lab.model == b, h].mean()
            rows.append({
                "judge": judge, "pair": f"{a} vs {b}", "type": h,
                "name": HNAME[h],
                "within_judge": f"{a}" if wa > wb else f"{b}",
                "within_gap_pp": round(100 * abs(wa - wb), 1),
                "aggregated": f"{a}" if ca > cb else f"{b}",
                "agrees": int((wa > wb) == (ca > cb)),
            })
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT_DIR / "judge_confound.csv", index=False)
out.append(cmp[["judge", "pair", "type", "name", "within_judge",
                "within_gap_pp", "aggregated", "agrees"]].to_markdown(index=False))

rate = 100 * cmp.agrees.mean()
out.append(f"\n\n**Orderings preserved: {cmp.agrees.sum()} of {len(cmp)} "
           f"({rate:.0f}%).**\n")
by_h = cmp.groupby("type").agrees.agg(["sum", "count"])
out.append("\nBy type:\n")
out.append(by_h.to_markdown())

# ------------------------------------- 5. does the axis signature hold
out.append("\n\n## 5. Dominant axis, within judge\n")
out.append("Axis rate = mean per-type prevalence within the axis, the "
           "aggregation used for the cross-generation check in the source "
           "study.\n")
AX = {"fabrication": ["H1", "H5", "H6"], "omission": ["H3"],
      "distortion": ["H2", "H4"]}
rows2 = []
for judge in sorted(raw.judge.unique()):
    for gen in sorted(raw.loc[raw.judge == judge, "model"].unique()):
        s = raw[(raw.judge == judge) & (raw.model == gen)]
        r = {"judge": judge, "generator": gen}
        for ax, cs in AX.items():
            r[ax] = round(100 * s[cs].mean().mean(), 1)
        r["dominant"] = max(AX, key=lambda a: r[a])
        rows2.append(r)
ax_df = pd.DataFrame(rows2)
out.append(ax_df.to_markdown(index=False))

out.append("\n\n### Aggregated dominant axis, for comparison\n")
rows3 = []
for gen in sorted(lab.model.unique()):
    s = lab[lab.model == gen]
    r = {"generator": gen}
    for ax, cs in AX.items():
        r[ax] = round(100 * s[cs].mean().mean(), 1)
    r["dominant"] = max(AX, key=lambda a: r[a])
    rows3.append(r)
out.append(pd.DataFrame(rows3).to_markdown(index=False))

(OUT_DIR / "JUDGE_CONFOUND.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


## 3. Rebuild labels

Verification gate. If ANY is not 91.1% and the mean is not 2.58, stop here and
check the inputs before trusting anything downstream.

In [ ]:
!python3 00_build_benchmark.py

## 4. Baselines, transfer, diagnostics

Step 2 is the slow one, roughly 10 to 15 minutes.

In [ ]:
!python3 01_baselines.py
!python3 02_logo_grounded.py
!python3 03_artifact_test.py
!python3 05_judge_confound.py
!python3 04_gt_audit.py

## 5. Bootstrap confidence intervals

The draft reports point estimates only. This is the number a reviewer will
demand, and the one result that could force a rewrite: if the
fabrication-versus-omission gap under generator shift has overlapping
intervals, finding 2 weakens from "inverts" to "converges".

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

H = ['H1','H2','H3','H4','H5','H6']; T = H + ['any_hallucination']
AX = {'fabrication':['H1','H5','H6'], 'omission':['H3'], 'distortion':['H2','H4']}

df = pd.read_csv(f'{OUT}/benchmark_labels.csv.gz')
df['model_output'] = df.model_output.fillna('')
X = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=5,
                    sublinear_tf=True, strip_accents='unicode'
                    ).fit_transform(df.model_output)

def boot(y, p, n=1000, seed=42):
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); out = []
    for _ in range(n):
        b = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y[b])) < 2: continue
        out.append(roc_auc_score(y[b], p[b]))
    return np.percentile(out, [2.5, 97.5])

rows, preds = [], {}
for held in ['Claude','GPT','Gemini']:
    tr = (df.model != held).to_numpy(); te = (df.model == held).to_numpy()
    for t in T:
        y = df[t].to_numpy()
        if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2: continue
        clf = LogisticRegression(max_iter=2000, class_weight='balanced',
                                 random_state=42).fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:,1]
        preds[(held,t)] = (y[te], p)
        lo, hi = boot(y[te], p)
        rows.append({'held_out':held, 'target':t,
                     'auc':round(roc_auc_score(y[te], p),3),
                     'ci_low':round(lo,3), 'ci_high':round(hi,3)})
    print('fold done:', held)

ci = pd.DataFrame(rows); ci.to_csv(f'{OUT}/logo_auc_ci.csv', index=False)
print()
print(ci.to_markdown(index=False))

### The decisive comparison

Pools the three folds and bootstraps the **gap** between the fabrication and
omission axes directly, rather than eyeballing two separate intervals. If this
interval excludes zero, finding 2 stands as written.

In [ ]:
rng = np.random.default_rng(42)
gaps = []
for _ in range(1000):
    fab, omi = [], []
    for held in ['Claude','GPT','Gemini']:
        for t in AX['fabrication']:
            y, p = preds[(held,t)]
            b = rng.choice(len(y), len(y), replace=True)
            if len(np.unique(y[b])) > 1: fab.append(roc_auc_score(y[b], p[b]))
        for t in AX['omission']:
            y, p = preds[(held,t)]
            b = rng.choice(len(y), len(y), replace=True)
            if len(np.unique(y[b])) > 1: omi.append(roc_auc_score(y[b], p[b]))
    gaps.append(np.mean(omi) - np.mean(fab))

lo, hi = np.percentile(gaps, [2.5, 97.5])
print(f'omission minus fabrication, held-out generator: '
      f'{np.mean(gaps):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')
print('EXCLUDES ZERO -> finding 2 holds' if lo > 0 or hi < 0
      else 'INCLUDES ZERO -> soften finding 2 in the abstract and 5.2')

## 6. Figures

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':9, 'figure.dpi':140, 'savefig.bbox':'tight'})

base = pd.read_csv(f'{OUT}/baseline_results.csv')
logo = pd.read_csv(f'{OUT}/logo_results.csv')
LBL  = ['H1\nScene Fab','H2\nMisclass','H3\nMissed',
        'H4\nSeverity','H5\nEntity Fab','H6\nPhantom']

# Fig 1: the inversion, with CIs
rnd = base[(base.features=='tfidf') & (base.split=='random')].set_index('target').auc
lg  = logo[logo.features=='tfidf'].groupby('target').auc.mean()
cim = ci.groupby('target')[['auc','ci_low','ci_high']].mean()
x = np.arange(6); w = 0.38
fig, ax = plt.subplots(figsize=(5.5,2.6))
ax.bar(x-w/2, [rnd[h] for h in H], w, label='same generator')
ax.bar(x+w/2, [lg[h] for h in H], w, label='held-out generator',
       yerr=[[lg[h]-cim.ci_low[h] for h in H],[cim.ci_high[h]-lg[h] for h in H]],
       capsize=2, ecolor='0.3')
ax.axhline(0.5, ls=':', c='gray', lw=.8)
ax.set_xticks(x); ax.set_xticklabels(LBL); ax.set_ylabel('AUC')
ax.set_ylim(0.45,0.95); ax.legend(frameon=False, ncol=2)
fig.savefig(f'{OUT}/fig1_inversion.pdf'); plt.show()

# Fig 2: feature sets by axis
piv = logo.pivot_table(index='features', columns='target', values='auc')
order = [f for f in ['surface','tfidf','embed','grounded','grounded+surface']
         if f in piv.index]
fig, ax = plt.subplots(figsize=(5.5,2.6))
for a, cols in AX.items():
    ax.plot(order, [piv.loc[f, cols].mean() for f in order], marker='o', label=a)
ax.plot(order, [piv.loc[f,'any_hallucination'] for f in order],
        marker='s', ls='--', c='k', label='ANY')
ax.axhline(0.5, ls=':', c='gray', lw=.8)
ax.set_ylabel('AUC (held-out generator)')
ax.legend(frameon=False, fontsize=7); plt.xticks(rotation=15)
fig.savefig(f'{OUT}/fig2_featuresets.pdf'); plt.show()

# Fig 3: label-source divergence
art = pd.read_csv(f'{OUT}/artifact_test.csv')
art = art[art.feature=='n_words']
fig, ax = plt.subplots(figsize=(4.0,2.8))
ax.scatter(art.AUC_vs_human, art.AUC_vs_panel, s=30, zorder=3)
for _, r in art.iterrows():
    ax.annotate(r['type'], (r.AUC_vs_human, r.AUC_vs_panel),
                textcoords='offset points', xytext=(5,3), fontsize=7)
lim = [0.2,0.9]; ax.plot(lim, lim, ls=':', c='gray', lw=.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('AUC of report length vs HUMAN label')
ax.set_ylabel('vs PANEL label')
fig.savefig(f'{OUT}/fig3_label_divergence.pdf'); plt.show()
print('figures ->', OUT)

## 7. Send back

Everything is in the `OUT` folder on Drive. Paste these into chat:

- the output of cell 3 (the verification gate)
- `logo_auc_ci.csv` and the gap interval from cell 5
- `BASELINE_REPORT.md`, `LOGO_REPORT.md`, `ARTIFACT_TEST.md`, `JUDGE_CONFOUND.md`

Any number that differs from the draft gets corrected in the paper.

In [ ]:
import os
for f in sorted(os.listdir(OUT)):
    print(f'{os.path.getsize(os.path.join(OUT,f))/1e3:10.1f} KB  {f}')